# Step 10-B1: LLM chấm thứ hai (Qwen2.5-7B-Instruct, vLLM) trên 4 bộ hiện có

Lặp lại ablation 2×2 (prompt paper/code × xáo/cố định) và đo thiên lệch vị trí với một LLM mã nguồn mở, để xem phát hiện 3 có phải chỉ là đặc tính của gpt-4o-mini.

- **Accelerator:** GPU T4 ×2. **Internet:** On. **Secret:** `HF_TOKEN` (như Step 7).
- **Input:** `qudar-beir-data`, `qudar-runs`, `qudar-patches` (bản mới có `llm_score.py --backend vllm`). **Không** gắn `qudar-llm` hay `qudar-llm-ablation` vào notebook này: tên file cache trùng nhau.
- **Thời gian ước tính:** cài đặt 5 phút; 3 lần nạp mô hình × 2–3 phút; khoảng 24.600 prompt ở 8–12 prompt/giây → 35–50 phút. Ô DAT (tùy chọn) thêm khoảng 10 phút. Tổng khoảng 1–1,5 giờ.
- **Sau khi chạy:** Save Version → tạo dataset `qudar-llm-qwen` từ thư mục `llm_qwen/`.

In [1]:
import glob, os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
DATA  = "/kaggle/input/datasets/karrychuong/qudar-beir-data/data"
PATCH = os.path.dirname(glob.glob("/kaggle/input/**/llm_score.py", recursive=True)[0])
RUNS  = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/fiqa_test/retrieval_result.json", recursive=True)[0]))
print("PATCH =", PATCH, "\nRUNS =", RUNS, os.listdir(RUNS))
assert "--backend" in open(f"{PATCH}/llm_score.py").read(), "qudar-patches chưa phải bản mới"
!pip install -q vllm pytrec-eval-terrier
!pip install -q --no-deps beir
!git clone -q https://github.com/kaist-dmlab/QuDAR.git /kaggle/working/QuDAR
# Không pip install QuDAR: chỉ cần thư mục src, tránh đụng phiên bản torch/transformers của vLLM
L = (f"python {PATCH}/llm_score.py --repo /kaggle/working/QuDAR --runs {RUNS} --data-root {DATA} "
     f"--out /kaggle/working/llm_qwen --backend vllm --model Qwen/Qwen2.5-7B-Instruct")

PATCH = /kaggle/input/datasets/karrychuong/qudar-patches/qudar-patches 
RUNS = /kaggle/input/datasets/karrychuong/qudar-runs/runs ['climate-fever_test', 'fiqa_test', 'fiqa_train', 'hotpotqa_test', 'hotpotqa_train', 'scidocs_test']
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
litellm 1.109.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba

In [2]:
# A. Ablation 2x2 đủ query trên FiQA, SciDocs, Climate-FEVER (4 chế độ, 1 lần nạp mô hình)
!{L} --mode paper code paper_fixed code_shuffle --names fiqa_test scidocs_test climate-fever_test

W1123 02:58:47.572000 90 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1123 02:58:47.573000 90 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1123 02:58:47.574000 90 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=90) INFO 11-23 02:59:12 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 6144, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 2.70MB/s]
(MainProcess pid=90) INFO 11-23 02:59:29 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=90) WARNING 11-23 02:59:29 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=90) INFO 11-23 02:59:29 [model.py:2214] Using max model len 6144
(MainProcess pid=90) INFO 11-23 0

In [3]:
# B. QuDAR-llm (chế độ paper) trên toàn bộ 7.405 query HotpotQA, cho bảng chính
!{L} --mode paper --names hotpotqa_test

W1123 04:55:40.972000 547 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1123 04:55:40.972000 547 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1123 04:55:40.973000 547 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=547) INFO 11-23 04:56:00 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 6144, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
(MainProcess pid=547) INFO 11-23 04:56:00 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=547) WARNING 11-23 04:56:00 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=547) INFO 11-23 04:56:00 [model.py:2214] Using max model len 6144
(MainProcess pid=547) INFO 11-23 04:56:00 [scheduler.py:354] Chunked prefill is enabled with max_num_batche

In [4]:
# C. Ablation trên 1.500 query đầu của HotpotQA như Step 6 (chế độ paper lấy từ cache của ô B)
!{L} --mode paper code paper_fixed code_shuffle --names hotpotqa_test --max-queries 1500

W1123 05:30:56.310000 803 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1123 05:30:56.310000 803 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1123 05:30:56.311000 803 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=803) INFO 11-23 05:31:19 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 6144, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
(MainProcess pid=803) INFO 11-23 05:31:19 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=803) WARNING 11-23 05:31:19 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=803) INFO 11-23 05:31:19 [model.py:2214] Using max model len 6144
(MainProcess pid=803) INFO 11-23 05:31:19 [scheduler.py:354] Chunked prefill is enabled with max_num_batche

In [5]:
# D. (Tùy chọn) DAT với Qwen chấm
!{L} --mode dat --names fiqa_test scidocs_test climate-fever_test hotpotqa_test

W1123 05:52:31.407000 1017 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1123 05:52:31.408000 1017 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1123 05:52:31.408000 1017 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=1017) INFO 11-23 05:52:52 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 6144, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
(MainProcess pid=1017) INFO 11-23 05:52:52 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=1017) WARNING 11-23 05:52:52 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=1017) INFO 11-23 05:52:52 [model.py:2214] Using max model len 6144
(MainProcess pid=1017) INFO 11-23 05:52:52 [scheduler.py:354] Chunked prefill is enabled with max_nu

In [6]:
# Tóm tắt để gửi lại: chỉ số + điểm trung bình theo vị trí hiển thị (cùng cách tính với step5_analysis.py)
import json, random, pandas as pd, numpy as np
from pathlib import Path
ev = pd.read_csv("/kaggle/working/llm_qwen/llm_eval.csv").drop_duplicates(["name", "method"], keep="last")
print(ev[["name", "method", "queries", "failed", "Recall@10", "nDCG@10"]].to_string(index=False))
rows = []
for f in sorted(Path("/kaggle/working/llm_qwen").glob("llm_*_*.jsonl")):
    mode = next((m for m in ("paper_fixed", "code_shuffle", "paper", "code") if f.name.startswith(f"llm_{m}_")), None)
    if mode is None:
        continue
    pos = {i: [] for i in range(1, 5)}
    for line in f.read_text().splitlines():
        r = json.loads(line)
        if not r["scores"]:
            continue
        order = ["ES", "OS", "ED", "OD"]
        if mode in ("paper", "code_shuffle"):
            order = ["OS", "OD", "ES", "ED"]
            random.Random(r["qid"]).shuffle(order)
        for i, s in enumerate(order, 1):
            pos[i].append(r["scores"][s])
    rows.append({"file": f.stem, **{f"pos {i}": round(float(np.mean(v)), 2) for i, v in pos.items()}})
print(pd.DataFrame(rows).to_string(index=False))

                   name                 method  queries  failed  Recall@10  nDCG@10
              fiqa_test              qudar_llm      648       0     0.3201   0.2469
              fiqa_test   qudar_llm_codeprompt      648       0     0.3360   0.2617
              fiqa_test  qudar_llm_paper_fixed      648       0     0.3058   0.2398
              fiqa_test qudar_llm_code_shuffle      648       0     0.3423   0.2601
           scidocs_test              qudar_llm     1000       0     0.1877   0.1809
           scidocs_test   qudar_llm_codeprompt     1000       0     0.1893   0.1813
           scidocs_test  qudar_llm_paper_fixed     1000       0     0.1889   0.1799
           scidocs_test qudar_llm_code_shuffle     1000       0     0.1900   0.1811
     climate-fever_test              qudar_llm     1535       0     0.2440   0.2023
     climate-fever_test   qudar_llm_codeprompt     1535       0     0.2388   0.1978
     climate-fever_test  qudar_llm_paper_fixed     1535       0     0.2352  